# Évaluation formelle

## Environnement et dépendances

In [ ]:
import os
import zipfile
from pathlib import Path
import yaml
import shutil
from tqdm import tqdm

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
    print("Environnement Google Colab détecté.")
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/General/Projet_AgriTG/AgbleDɔ01')
    os.chdir(PROJECT_ROOT)
    !pip install -q ultralytics mlflow

except ImportError:
    IN_COLAB = False
    print("Environnement local détecté.")
    PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
    os.chdir(PROJECT_ROOT)

print(f"Racine du projet : {PROJECT_ROOT}")

Environnement Google Colab détecté.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Racine du projet : /content/drive/MyDrive/General/Projet_AgriTG/AgbleDɔ01


In [ ]:
import mlflow
from ultralytics import YOLO, settings

MODEL_PATH = PROJECT_ROOT / "models" / "agbledɔ01.pt"
DATA_DIR = PROJECT_ROOT / "data" / "processed"

settings.update({"mlflow": False})
model = YOLO(MODEL_PATH)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
if IN_COLAB:

    # ========= Config =========
    drive_zip = str(PROJECT_ROOT / "data" / "processed.zip")
    colab_zip = "/content/processed.zip"
    COLAB_DATA_DIR = "/content/data"

    # ========= Utils =========
    def zip_valid(zip_path):
        try:
            with zipfile.ZipFile(zip_path, "r") as z:
                return z.testzip() is None
        except:
            return False

    def total_size(path):
        return sum(
            os.path.getsize(os.path.join(root, f))
            for root, _, files in os.walk(path)
            for f in files
        )

    # ========= Build zip if needed =========
    if not os.path.exists(drive_zip) or not zip_valid(drive_zip):

        print("\n📦 Création du zip...")

        if os.path.exists(drive_zip):
            os.remove(drive_zip)

        with zipfile.ZipFile(
            drive_zip,
            "w",
            zipfile.ZIP_DEFLATED
        ) as z:

            for root, _, files in os.walk(str(DATA_DIR)):
                for file in tqdm(files):
                    full = os.path.join(root, file)
                    rel = os.path.relpath(full, str(DATA_DIR))
                    z.write(full, rel)

        if not zip_valid(drive_zip):
            raise Exception("❌ ZIP invalide")

        print("[OK] - ZIP OK")

    else:
        print("\n[OK] - ZIP déjà valide")

    # ========= Copy =========
    print("\n🚚 Copie vers /content...")

    if os.path.exists(colab_zip):
        os.remove(colab_zip)

    shutil.copy2(drive_zip, colab_zip)
    print("[OK] - Copie")

    # ========= Extract =========
    print("\n📂 Extraction...")

    if os.path.exists(COLAB_DATA_DIR):
        shutil.rmtree(COLAB_DATA_DIR)

    os.makedirs(COLAB_DATA_DIR, exist_ok=True)
    with zipfile.ZipFile(colab_zip, "r") as z:
        z.extractall(COLAB_DATA_DIR)

    print("[OK] - Extraction")

    # ========= Verify =========
    print("\n🔎 Vérification...")

    src_count = sum(
        len(files)
        for _, _, files in os.walk(str(DATA_DIR))
    )

    dst_count = sum(
        len(files)
        for _, _, files in os.walk(COLAB_DATA_DIR)
    )

    src_size = total_size(str(DATA_DIR))
    dst_size = total_size(COLAB_DATA_DIR)

    assert src_count == dst_count, "❌ Nombre fichiers différent"
    assert src_size == dst_size, "❌ Taille dataset différente"

    print("[OK] - Vérification")

    # ========= Cleanup =========
    os.remove(colab_zip)
    DATA_DIR = Path(COLAB_DATA_DIR)
    print("\n [OK] - Dataset prêt :", DATA_DIR)


📦 Création du zip...


100%|██████████| 1/1 [00:00<00:00,  3.54it/s]
0it [00:00, ?it/s]
100%|██████████| 843/843 [00:20<00:00, 41.08it/s] 
0it [00:00, ?it/s]
100%|██████████| 298/298 [00:05<00:00, 57.86it/s] 


[OK] - ZIP OK

🚚 Copie vers /content...
[OK] - Copie

📂 Extraction...
[OK] - Extraction

🔎 Vérification...
[OK] - Vérification

 [OK] - Dataset prêt : /content/data


## Eval sur le test set

In [ ]:
metrics = model.val(
    split='test',
    project="agbledɔ01-yolo_experiment-01",
    name='agbledɔ01-eval-01'
)

print(f"Top-1 Accuracy : {metrics.top1:.4f} ({metrics.top1 * 100:.2f}%)")
print(f"Top-5 Accuracy : {metrics.top5:.4f} ({metrics.top5 * 100:.2f}%)")

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (AMD EPYC 7B12)
YOLO11s-cls summary (fused): 47 layers, 5,446,938 parameters, 0 gradients, 12.0 GFLOPs
train: /content/data/train... found 9838 images in 10 classes ✅ 
val: /content/data/val... found 2108 images in 10 classes ✅ 
test: /content/data/test... found 2109 images in 10 classes ✅ 
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1719.3±747.9 MB/s, size: 51.6 KB)
test: Scanning /content/data/test... 2105 images, 4 corrupt: 100% ━━━━━━━━━━━━ 2109/2109 5.3Kit/s 0.4s
test: /content/data/test/Corn___Gray_Leaf_Spot/Corn___Gray_Leaf_Spot_CCMT_01602.jpg: ignoring corrupt image/label: broken data stream when reading image file
test: /content/data/test/Corn___Healthy/Corn___Healthy_CCMT_01220.jpg: ignoring corrupt image/label: broken data stream when reading image file
test: /content/data/test/Corn___Northern_Leaf_Blight/Corn___Northern_Leaf_Blight_CCMT_01959.jpg: ignoring corrupt image/label: broken data stream when reading im